# INTERNACIONES

In [1]:
# Control de actualizacion de los cubos de internaciones
from mstrio.api import cubes
from utils.login import conn

cubo_seg_int = '5A0C8E704B69D749E96C4AB8911C2555'
cubo_base_cir = 'C74D6DAC5242DE0D4CBD0DBA74262207'

for cubo in [cubo_seg_int, cubo_base_cir]:
    info_cubo = cubes.cube_info(conn, cubo)
    resultado = info_cubo.json()
    print(f'✨ {resultado["cubesInfos"][0]["cubeName"]} fue actualizado por última vez el {resultado["cubesInfos"][0]["lastUpdateTime"]}')

Connection to Strategy One Intelligence Server has been established.
Project selected in Connection object:
Project object named: 'Salud' with ID: 'DAE6DF9811D67BD9500010A51D1D2ADA'
✨ 1.1 Cubo_Seguimiento_Internaciones fue actualizado por última vez el 6/30/2026 07:20:47
✨ 1.2 Base_Cirugias.csv fue actualizado por última vez el 6/30/2026 07:23:13


In [2]:
# Cargar bases de datos de Internaciones 🗄️
%run utils/bases_internacion.py
s = sanatoriales # type: ignore
q = quirurgicas # type: ignore

Connection to Strategy One Intelligence Server has been established.
Project selected in Connection object:
Project object named: 'Salud' with ID: 'DAE6DF9811D67BD9500010A51D1D2ADA'
Report object named: 'Cantidades & Aut. Valorizadas' with ID: '9893BB69E24DAAE8C6814681ADE6931A'
Report object named: 'NE & Presupuesto' with ID: '7372D310AB471232EF5C7EAB8D787F24'
Report object named: 'Cantidades (S)' with ID: 'BE1E34836C4AEB490E0B33B7B016945E'
Report object named: 'NE (S)' with ID: '1994C345F8494B6D7313339B24BB6CF6'


In [3]:
# Actualizar archivo Excel con nuevas hojas 📊
import pandas as pd
import os

# Cargando fecha de hoy y periodo actual ⏳
hoy = pd.Timestamp.today().normalize() # Fecha de hoy
dia_hoy = hoy.strftime('%Y-%m-%d') # Fecha de hoy en formato string 'YYYY-MM-DD'
print(f"Fecha de hoy: {dia_hoy}")

# Cargar el archivo existente
file_path = f'files/gsheet_{dia_hoy}.xlsx'

# Verificar si el archivo existe
if os.path.exists(file_path):
    mode = 'a'  # append si existe
else:
    print(f"El archivo {file_path} no existe. Por favor, crea el archivo primero.")
    exit()

with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    s.to_excel(writer, sheet_name='Int_S', index=False)
    q.to_excel(writer, sheet_name='Int_Q', index=False)
    
print("Archivo Excel actualizado exitosamente. 🆗")

Fecha de hoy: 2026-06-30
Archivo Excel actualizado exitosamente. 🆗


In [4]:
import pandas as pd
import os
import re

# 1. Define la ruta de tu carpeta
ruta_carpeta = r'C:\Users\maalsosa\Documents\scripts\SdA\SdA 2025-2026\files' 

# Listas separadas para cada tipo de hoja
datos_int_s = []
datos_int_q = []

print("--- Iniciando proceso ---")

# 2. Loop para leer los archivos
for archivo in os.listdir(ruta_carpeta):
    if archivo.endswith(".xlsx") or archivo.endswith(".xls"):
        
        # Extraemos la fecha del nombre del archivo
        match_fecha = re.search(r'(\d{4}-\d{2}-\d{2})', archivo)
        fecha_archivo = match_fecha.group(1) if match_fecha else "Sin Fecha"

        ruta_completa = os.path.join(ruta_carpeta, archivo)
        
        try:
            xls = pd.ExcelFile(ruta_completa)
            hojas_presentes = xls.sheet_names
            
            # --- Procesamos Int_S ---
            if 'Int_S' in hojas_presentes:
                print(f"✅ {archivo}: Leyendo Int_S...")
                df_s = pd.read_excel(xls, sheet_name='Int_S')
                df_s['Fecha_Archivo'] = fecha_archivo  # Agregamos la fecha
                datos_int_s.append(df_s)
            else:
                print(f"⚠️ {archivo}: No tiene Int_S")

            # --- Procesamos Int_Q ---
            if 'Int_Q' in hojas_presentes:
                print(f"✅ {archivo}: Leyendo Int_Q...")
                df_q = pd.read_excel(xls, sheet_name='Int_Q')
                df_q['Fecha_Archivo'] = fecha_archivo  # Agregamos la fecha
                datos_int_q.append(df_q)
            else:
                print(f"⚠️ {archivo}: No tiene Int_Q")
                
        except Exception as e:
            print(f"❌ Error leyendo {archivo}: {e}")

print("\n" + "="*30)

# 3. Consolidación en dos tablas distintas
# Tabla final para Int_S
if datos_int_s:
    df_final_s = pd.concat(datos_int_s, ignore_index=True)
    print(f"Tabla 'Int_S' consolidada: {len(df_final_s)} filas.")
    # df_final_s.to_excel("Consolidado_Int_S.xlsx", index=False) 
else:
    df_final_s = pd.DataFrame() # DataFrame vacío si no hubo datos
    print("No se encontraron datos para Int_S.")

# Tabla final para Int_Q
if datos_int_q:
    df_final_q = pd.concat(datos_int_q, ignore_index=True)
    print(f"Tabla 'Int_Q' consolidada: {len(df_final_q)} filas.")
    # df_final_q.to_excel("Consolidado_Int_Q.xlsx", index=False)
else:
    df_final_q = pd.DataFrame()
    print("No se encontraron datos para Int_Q.")

# Ahora tienes dos variables disponibles para usar:
# df_final_s -> Contiene todo lo de Int_S
# df_final_q -> Contiene todo lo de Int_Q

--- Iniciando proceso ---
⚠️ gsheet_2025-09-17.xlsx: No tiene Int_S
⚠️ gsheet_2025-09-17.xlsx: No tiene Int_Q
⚠️ gsheet_2025-09-18.xlsx: No tiene Int_S
⚠️ gsheet_2025-09-18.xlsx: No tiene Int_Q
⚠️ gsheet_2025-09-19.xlsx: No tiene Int_S
⚠️ gsheet_2025-09-19.xlsx: No tiene Int_Q
⚠️ gsheet_2025-09-22.xlsx: No tiene Int_S
⚠️ gsheet_2025-09-22.xlsx: No tiene Int_Q
⚠️ gsheet_2025-09-23.xlsx: No tiene Int_S
⚠️ gsheet_2025-09-23.xlsx: No tiene Int_Q
⚠️ gsheet_2025-09-24.xlsx: No tiene Int_S
⚠️ gsheet_2025-09-24.xlsx: No tiene Int_Q
⚠️ gsheet_2025-09-25.xlsx: No tiene Int_S
⚠️ gsheet_2025-09-25.xlsx: No tiene Int_Q
⚠️ gsheet_2025-09-26.xlsx: No tiene Int_S
⚠️ gsheet_2025-09-26.xlsx: No tiene Int_Q
⚠️ gsheet_2025-09-29.xlsx: No tiene Int_S
⚠️ gsheet_2025-09-29.xlsx: No tiene Int_Q
⚠️ gsheet_2025-10-02.xlsx: No tiene Int_S
⚠️ gsheet_2025-10-02.xlsx: No tiene Int_Q
⚠️ gsheet_2025-10-03.xlsx: No tiene Int_S
⚠️ gsheet_2025-10-03.xlsx: No tiene Int_Q
⚠️ gsheet_2025-10-06.xlsx: No tiene Int_S
⚠️ gshee

In [5]:
# Usamos ExcelWriter para gestionar múltiples hojas
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    
    # Guardamos la tabla S si tiene datos
    if not df_final_s.empty:
        df_final_s.to_excel(writer, sheet_name='Consolidado_Int_S', index=False)
        print("✅ Hoja 'Consolidado_Int_S' guardada.")
    
    # Guardamos la tabla Q si tiene datos
    if not df_final_q.empty:
        df_final_q.to_excel(writer, sheet_name='Consolidado_Int_Q', index=False)
        print("✅ Hoja 'Consolidado_Int_Q' guardada.")

print(f"\nArchivo guardado exitosamente como: {file_path}")

✅ Hoja 'Consolidado_Int_S' guardada.
✅ Hoja 'Consolidado_Int_Q' guardada.

Archivo guardado exitosamente como: files/gsheet_2026-06-30.xlsx
